In [ ]:
import re
import fitz  # PyMuPDF
import pandas as pd
import pdfplumber


# 1. Extraire toutes les informations

## 1.1 Extraire toutes les questions
Une question est défini par une liste de caractères en gras qui finit par :


In [ ]:
pdf_path = "data/utils/downloaded_files/r23-474-annexe3.pdf"

In [ ]:
def extract_questions_from_pdf(pdf_path: str) -> list:
    """Extract all bold questions ending with ":" from a PDF.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        list: Sorted list of unique questions.
    """
    results = []
    # Using fitz to extract text with font information
    doc = fitz.open(pdf_path)
    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            # Only consider blocks that contain lines of text
            if "lines" not in block:
                continue
            for line in block["lines"]:
                # spans is a list of text segments with the same font properties
                for span in line["spans"]:
                    text = span["text"].strip()
                    if "bold" in span["font"].lower() and text.endswith(":"):
                        results.append(text)
    cleaned = [
        re.sub(r"\s+", " ", q).rstrip(":").strip() for q in results if q.strip() != ":"
    ]
    return sorted(set(cleaned))

## 1.2 Extraire les réponses

### 1.2.1 Lecture du PDF

In [ ]:
def read_pdf(path: str) -> str:
    """Read the full text from a PDF file.

    Args:
        path (str): Path to the PDF file.

    Returns:
        str: Full text of the PDF as a single string.
    """
    with pdfplumber.open(path) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

### 1.2.2 Récupération des blocs liés aux bâtiments
- Regex qui cherche 
    - 2 chiffres pour le code (2A et 2B aussi pour la corse sur ce fichier il y en a pas)
    - un tiret
    - une liste de caractères 
exemple '01 - Ain'
- Création des blocks

In [ ]:
def split_departments(text: str) -> list[tuple[str, str]]:
    """Split the PDF text into department blocks.

    Args:
        text (str): Full text of the PDF.

    Returns:
        list[tuple[str, str]]: List of tuples (department_name, department_text_block).
    """
    # Regex: entre 2 et 3 chiffres (ex: 01, 971, 2A, 2B), tiret, puis nom du département (sur une seule ligne)
    dept_pattern = re.compile(r"^(?:\d{2,3}|2A|2B) - [^\n]+$", re.MULTILINE)

    matches = list(dept_pattern.finditer(text))

    blocks = []
    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        blocks.append((matches[i].group().strip(), text[start:end]))
    return blocks

### 1.2.3 Récupération des informations liés aux collectivités
- On sépare les blocs liés aux collectivités
- On vient chercher les réponse aux questions :
    Pour "Solutions proposées pour régler les problèmes d’assurance des collectivités" on vient prendre toutes les données jusqu'à une nouvelle collectivitée ou nouveau département


In [ ]:
def split_collectivities(block: str) -> list[str]:
    """Split a department block into individual collectivities.

    Args:
        block (str): Text block of a department.

    Returns:
        list[str]: List of text blocks, each representing a collectivity.
    """
    return [
        c.strip() for c in re.split(r"(?m)^Collectivité\s*:\s*", block)[1:] if c.strip()
    ]


def parse_collectivity(text: str, department: str, questions: list[str]) -> dict:
    """Parse a single collectivity and extract answers to questions.

    Only the question
    "Solutions proposées pour régler les problèmes d’assurance des collectivités"
    allows multi-line responses until the next collectivity or department.

    Args:
        text (str): Text of the collectivity.
        department (str): Name of the department.
        questions (list[str]): List of questions to extract.

    Returns:
        dict: Dictionary with department, collectivity, and question responses.
    """
    record = {"department": department, "collectivity": text.split("\n")[0].strip()}

    for q in questions:
        if (
            q
            == "Solutions proposées pour régler les problèmes d’assurance des collectivités"
        ):
            pattern = re.compile(rf"(?ms){re.escape(q)}\s*:\s*(.*)")
        else:
            pattern = re.compile(rf"(?m)^{re.escape(q)}\s*:\s*(.+)$")

        match = pattern.search(text)
        record[q] = match.group(1).strip() if match else None

    return record

In [ ]:
def parse_department(block: str, name: str, questions: list[str]) -> list[dict]:
    """Parse all collectivities in a department block.

    Args:
        block (str): Department text block.
        name (str): Department name.
        questions (list[str]): List of questions to extract.

    Returns:
        list[dict]: List of dictionaries for each collectivity.
    """
    return [parse_collectivity(c, name, questions) for c in split_collectivities(block)]


def extract_information_from_pdf(pdf_path: str) -> pd.DataFrame:
    """Extract all information from a PDF into a DataFrame.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        pd.DataFrame: DataFrame with columns for department, collectivity, and questions.
    """
    text = read_pdf(pdf_path)
    questions = extract_questions_from_pdf(
        pdf_path
    )  # Your function to extract questions
    data = []

    for dept_name, dept_block in split_departments(text):
        data.extend(parse_department(dept_block, dept_name, questions))

    return pd.DataFrame(data)


In [ ]:
df_extract = extract_information_from_pdf(pdf_path=pdf_path)

# 2. Rendre les informations disponibles

## 2.1 Créer la table collectivitées

In [ ]:
df_extract["id_collectivity"] = df_extract.index


In [ ]:
collectivities = pd.DataFrame()

collectivities["id_collectivity"] = df_extract["id_collectivity"]
collectivities["department_code"] = df_extract["department"].str.extract(
    r"^(\d{2,3}|2A|2B)"
)
collectivities["department_name"] = df_extract["department"].str.extract(
    r"^(?:\d{2,3}|2A|2B)\s*-\s*(.+)"
)
collectivities["collectivity"] = (
    df_extract["collectivity"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
)
collectivities["type_collectivity"] = df_extract["collectivity"].str.extract(
    r"\(([^()]*)\)\s*$"
)
collectivities["job_repondant"] = df_extract["Fonction"]
collectivities

## 2.2 Créer la table problem

In [ ]:
import re

events = []

for i, txt in df_extract["Problème(s) majeur(s) évoqué(s)"].dropna().items():
    parts = re.split(r"[-•]", txt)
    for p in parts:
        p = p.strip()
        if p and not p.isdigit():
            events.append({"id_collectivite": i, "problem": p})
problemes_solutions = pd.DataFrame(events)
problemes_solutions

## 2.3 Solutions

In [ ]:
note_solutions = pd.DataFrame()
note_solutions["solution"] = df_extract[
    "Solutions proposées pour régler les problèmes d’assurance des collectivités"
]
note_solutions["note_relation"] = (
    df_extract["Qualité de la relation avec l’assureur pour les dommages aux biens"]
    .str.replace("/10", "", regex=False)
    .astype(float)
)
note_solutions["id_collectivity"] = df_extract["id_collectivity"]
note_solutions = note_solutions[["id_collectivity", "solution", "note_relation"]]
